# shipments database for the garage door (simulated)

this notebook makes a small database of fake shipments for the dock door.
nothing here talks to the PLC yet. it only builds the data the door will use later.

the plan:
1. **this notebook**: a shipment schedule + a log of what happened at the door
2. **next**: the door notebook reads the schedule ("is a truck expected now?")
3. **after that**: YOLO watches a camera or video, sees a truck, and checks it against the schedule before the door opens

the database is one file, `dock.db`, in this folder. it uses `sqlite3`, which comes with python, so there is nothing to install.

## 1. setup

run this cell first. it opens (or creates) `dock.db`.

In [ ]:
import sqlite3
import random
from datetime import datetime, timedelta

DB_PATH = "dock.db"

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row   # lets us read columns by name
print("opened", DB_PATH)

## 2. create the tables

two tables:

**shipments** is the schedule. one row per truck we expect.

| column | meaning |
|---|---|
| shipment_id | e.g. SHP-1001 |
| carrier | trucking company (made up) |
| truck_id | trailer number on the side of the truck |
| direction | inbound (receiving) or outbound (shipping) |
| door | which dock door, we only have door 1 for now |
| scheduled_at | when the truck is due |
| pallets | how many pallets expected |
| status | scheduled → arrived → loading → done, or late / no_show |

**door_events** is the log. every time the door opens, closes, or a truck is seen, a row goes here. later the door notebook and YOLO will write to it.

running this cell twice is safe. it only creates tables that don't exist yet.

In [ ]:
conn.executescript("""
CREATE TABLE IF NOT EXISTS shipments (
    shipment_id   TEXT PRIMARY KEY,
    carrier       TEXT NOT NULL,
    truck_id      TEXT NOT NULL,
    direction     TEXT NOT NULL CHECK (direction IN ('inbound','outbound')),
    door          INTEGER NOT NULL DEFAULT 1,
    scheduled_at  TEXT NOT NULL,
    pallets       INTEGER NOT NULL,
    status        TEXT NOT NULL DEFAULT 'scheduled',
    arrived_at    TEXT,
    finished_at   TEXT
);

CREATE TABLE IF NOT EXISTS door_events (
    event_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    ts           TEXT NOT NULL,
    door         INTEGER NOT NULL DEFAULT 1,
    event        TEXT NOT NULL,   -- truck_seen, door_open, door_shut, door_stop, ...
    shipment_id  TEXT,            -- blank if we don't know which truck
    source       TEXT,            -- manual, plc, yolo
    note         TEXT
);
""")
conn.commit()
print("tables ready")

## 3. make fake shipments

this fills today's schedule with made-up trucks, roughly one every 45 to 90 minutes starting at 7:00.
`random.seed(42)` means you get the same trucks every time. change the number for a different day.

if you run it again it wipes the old schedule first, so you always start clean.

In [ ]:
CARRIERS = ["Tarheel Freight", "Neuse River Logistics", "Pinecone Transport",
            "Crabtree Carriers", "Triangle Haul"]

def make_fake_day(n_trucks=8, start_hour=7, seed=42):
    random.seed(seed)
    conn.execute("DELETE FROM shipments")
    t = datetime.now().replace(hour=start_hour, minute=0, second=0, microsecond=0)
    rows = []
    for i in range(n_trucks):
        t += timedelta(minutes=random.randint(45, 90))
        rows.append((
            f"SHP-{1001 + i}",
            random.choice(CARRIERS),
            f"TRL-{random.randint(100, 999)}",
            random.choice(["inbound", "inbound", "outbound"]),  # more receiving than shipping
            1,
            t.strftime("%Y-%m-%d %H:%M"),
            random.randint(4, 26),
            "scheduled",
        ))
    conn.executemany(
        """INSERT INTO shipments
           (shipment_id, carrier, truck_id, direction, door, scheduled_at, pallets, status)
           VALUES (?,?,?,?,?,?,?,?)""", rows)
    conn.commit()
    print(f"made {len(rows)} shipments")

make_fake_day()

## 4. look at the schedule

`pandas` makes it print as a nice table. if you get `No module named pandas`, run `%pip install pandas` in a new cell, then restart.

In [ ]:
import pandas as pd

def show(sql="SELECT * FROM shipments ORDER BY scheduled_at", params=()):
    return pd.read_sql_query(sql, conn, params=params)

show()

## 5. helper functions the door will use later

these are the pieces the door notebook (and later YOLO) will call:

- `next_expected()`: the next truck that hasn't arrived yet
- `find_truck(truck_id)`: look up a truck by trailer number (this is what YOLO + a plate/number reader would do)
- `mark_arrived(shipment_id)`, `mark_done(shipment_id)`: move a shipment along
- `log_event(event, ...)`: write a row to the door log

In [ ]:
def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def next_expected():
    return conn.execute(
        """SELECT * FROM shipments
           WHERE status = 'scheduled'
           ORDER BY scheduled_at LIMIT 1""").fetchone()

def find_truck(truck_id):
    return conn.execute(
        "SELECT * FROM shipments WHERE truck_id = ? AND status IN ('scheduled','late')",
        (truck_id,)).fetchone()

def log_event(event, shipment_id=None, source="manual", note=None, door=1):
    conn.execute(
        "INSERT INTO door_events (ts, door, event, shipment_id, source, note) VALUES (?,?,?,?,?,?)",
        (now_str(), door, event, shipment_id, source, note))
    conn.commit()

def mark_arrived(shipment_id, source="manual"):
    conn.execute("UPDATE shipments SET status='arrived', arrived_at=? WHERE shipment_id=?",
                 (now_str(), shipment_id))
    conn.commit()
    log_event("truck_arrived", shipment_id, source)

def mark_done(shipment_id, source="manual"):
    conn.execute("UPDATE shipments SET status='done', finished_at=? WHERE shipment_id=?",
                 (now_str(), shipment_id))
    conn.commit()
    log_event("shipment_done", shipment_id, source)

nxt = next_expected()
print("next truck:", dict(nxt) if nxt else "none")

## 6. pretend a truck shows up

this is the whole flow by hand, no PLC and no camera yet:

1. a truck pulls up, we read its trailer number
2. look it up in the schedule
3. if it's expected: mark it arrived and log "door should open"
4. if it isn't: log it as unknown, door stays shut

later, step 1 comes from YOLO and step 3 sends `open_pb` to the PLC.

In [ ]:
def truck_pulls_up(truck_id, source="manual"):
    log_event("truck_seen", source=source, note=f"trailer {truck_id}")
    s = find_truck(truck_id)
    if s is None:
        log_event("door_denied", source=source, note=f"{truck_id} not on schedule")
        print(f"{truck_id}: not on the schedule. door stays shut.")
        return False
    mark_arrived(s["shipment_id"], source)
    log_event("door_open_request", s["shipment_id"], source)
    print(f"{truck_id}: {s['shipment_id']} from {s['carrier']}, "
          f"{s['direction']}, {s['pallets']} pallets. open the door.")
    return True

# try it with the first scheduled truck
first = next_expected()
truck_pulls_up(first["truck_id"])

# and a truck nobody expected
_ = truck_pulls_up("TRL-000")

## 7. check the log and the schedule

In [ ]:
show("SELECT * FROM door_events ORDER BY event_id")

In [ ]:
show()

## 8. optional: save as a spreadsheet

writes `shipments.csv` and `door_events.csv` to this folder so you can open them in Numbers or Excel.
the database stays the source of truth. the csv files are just a snapshot.

In [ ]:
show().to_csv("shipments.csv", index=False)
show("SELECT * FROM door_events ORDER BY event_id").to_csv("door_events.csv", index=False)
print("saved shipments.csv and door_events.csv")

## 9. reset

run this when you want a fresh day: clears the log and rebuilds the schedule.

In [ ]:
conn.execute("DELETE FROM door_events")
conn.commit()
make_fake_day()
show()

## 10. open the door from python

this sends a short pulse to `remote_open` (`%MX0.2`) in OpenPLC, like tapping the open button.

before you run it:
- the openplc runtime is running (`docker start openplc-runtime`)
- door_sim.ipynb is running in its own tab and says `connected. door starts shut.`

the door should go up and keep going after the pulse ends (the seal-in holds it).
we pulse instead of holding it TRUE, because if `remote_open` stays on, stop won't stay stopped.

In [ ]:
from pymodbus.client import ModbusTcpClient
import os
import time

# 127.0.0.1 on your mac. docker compose sets PLC_HOST=openplc inside the container.
PLC_HOST = os.environ.get("PLC_HOST", "127.0.0.1")
PLC_PORT = int(os.environ.get("PLC_PORT", "1502"))
REMOTE_OPEN = 8194   # %MX0.2  (8192 + 2)

def open_door():
    plc = ModbusTcpClient(PLC_HOST, port=PLC_PORT)
    if not plc.connect():
        print("could not connect. is the openplc runtime running?")
        return False
    plc.write_coil(REMOTE_OPEN, True)    # press
    time.sleep(0.5)
    plc.write_coil(REMOTE_OPEN, False)   # release
    plc.close()
    print("pulsed remote_open")
    return True

_ = open_door()

## 11. a truck pulls up and the door opens

same as step 6, but now a truck on the schedule opens the door for real.
a truck that isn't on the schedule gets logged and the door stays shut.

run step 9 (reset) first if you want a clean schedule, and close the door in the debugger between tries
(the door only opens from shut or stopped, not while it's already open).

In [ ]:
def truck_at_door(truck_id, source="manual"):
    ok = truck_pulls_up(truck_id, source)     # checks the schedule and logs it
    if ok:
        if open_door():
            log_event("door_open_sent", find_shipment(truck_id), source)
    return ok

def find_shipment(truck_id):
    row = conn.execute(
        "SELECT shipment_id FROM shipments WHERE truck_id = ? ORDER BY arrived_at DESC LIMIT 1",
        (truck_id,)).fetchone()
    return row["shipment_id"] if row else None

# the next truck on the schedule: the door should open
nxt = next_expected()
_ = truck_at_door(nxt["truck_id"])

In [ ]:
# a truck nobody expected: the door should stay shut
_ = truck_at_door("TRL-000")

show("SELECT * FROM door_events ORDER BY event_id")